# 20 — FastText

**Goal:** Handle rare words and typos using subword information.

## 1. Training FastText with Gensim

In [ ]:
from gensim.models import FastText
sentences = [
    ["python", "deep", "learning", "framework"],
    ["nlp", "with", "python", "is", "powerful"],
    ["data", "science", "uses", "machine", "learning"],
    ["python", "programming", "language", "for", "data"],
]
ft = FastText(sentences, vector_size=50, window=3, min_count=1, epochs=100)
print(f"Vocabulary: {len(ft.wv)}")
print(f"Vector for 'python': shape {ft.wv['python'].shape}")

## 2. FastText Handles Out-of-Vocabulary Words

In [ ]:
# Word2Vec would fail here — FastText uses subword n-grams
oov_words = ["pythoning", "pyton", "tensorflo", "lernin", "progamming"]
for w in oov_words:
    try:
        vec = ft.wv[w]  # FastText can infer from subwords
        print(f"  '{w}' -> VECTOR FOUND (via subwords)")
    except KeyError:
        print(f"  '{w}' -> NOT FOUND")

# Compare with similar known word
for w in oov_words:
    if w in ft.wv:
        similar = ft.wv.most_similar(w, topn=2)
        print(f"  '{w}' similar to: {similar}")

## 3. FastText vs Word2Vec on Typos

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Correct vs misspelled
pairs = [("python", "pyton"), ("learning", "lernin"), ("tensorflow", "tensorflo")]
for correct, typo in pairs:
    if correct in ft.wv and typo in ft.wv:
        sim = cosine_similarity([ft.wv[correct]], [ft.wv[typo]])[0][0]
        print(f"  sim('{correct}', '{typo}') = {sim:.3f}")

# Compare with Word2Vec
print("\nWord2Vec would throw KeyError on typos.")
print("FastText gracefully handles misspellings via subword n-grams.")

## 4. Subword Details

In [ ]:
# FastText stores character n-grams
word = "python"
print(f"Subword n-grams for '{word}' (n=3):")
if hasattr(ft.wv, 'ngrams'):
    from gensim.models.fasttext import compute_ngrams
    ngrams = compute_ngrams(word, 3, 6)
    print(f"  {ngrams}")

# This means "pyton" shares many n-grams with "python" -> similar vectors
print("\nThis is why FastText handles typos — shared character sequences!")

## Summary: Use FastText for resume parsing — resumes often have typos ('Tensorflo', 'Pytorch').